In [13]:
import pandas as pd
import glob

# Load all CSVs from a folder
all_files = glob.glob("../data/cleaned_data/*.csv")
df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

print(f"Total rows: {len(df)}")

Total rows: 200020


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode categoricals
for col in df.select_dtypes(include="object").columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

TARGET = "risk_category"   # update to your target column
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                      scale_pos_weight=len(y[y==0])/len(y[y==1]))  # handles imbalance
model.fit(X_train, y_train)

In [12]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_prob):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")


Test Accuracy : 0.9975
ROC-AUC Score : 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     21023
           1       1.00      1.00      1.00     18981

    accuracy                           1.00     40004
   macro avg       1.00      1.00      1.00     40004
weighted avg       1.00      1.00      1.00     40004

